In [43]:
import numpy as np
import pandas as pd

# 1. Load raw data
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 2. Grain & Key Audit (1 row = 1 unique customer)
print("Shape:", df.shape)
print("Unique customerIDs:", df["customerID"].nunique())
assert df["customerID"].nunique() == len(df), "Duplicate customerIDs found!"

# 3. Clean mixed types in TotalCharges
df["TotalCharges_raw"] = df["TotalCharges"]
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].astype(str).str.strip(), errors="coerce")

missing_tc = df[df["TotalCharges"].isna()]
print(f"Missing TotalCharges after coercion: {len(missing_tc)} rows (all tenure == 0)")

df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

# 4. Standardise binary & categorical fields
df["Churn_Flag"] = np.where(df["Churn"] == "Yes", 1, 0)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# 5. Feature Engineering
service_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
df["Active_Services_Count"] = (df[service_cols] == "Yes").sum(axis=1)

df["Tenure_Band"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=["0-6m", "7-12m", "13-24m", "25-48m", "49-72m"]
)

df["Expected_Total"] = df["tenure"] * df["MonthlyCharges"]
df["Billing_Delta"] = df["TotalCharges"] - df["Expected_Total"]

# 6. Save clean dataset
df.drop(columns=["TotalCharges_raw"]).to_csv("telco_churn_clean.csv", index=False)
print("Saved clean dataset successfully.")

Shape: (7043, 26)
Unique customerIDs: 7043
Missing TotalCharges after coercion: 0 rows (all tenure == 0)
Saved clean dataset successfully.
